In [68]:
import pandas

## AppSongs

In [69]:
class AppSong:
    def __init__(self, title, filename):
        self.title = title
        self.filename = filename
        self.yt_link = None
        self.playlist_title = None
        self.page = None

    def __str__(self):
        return self.title
    
def read_songs(folder):
    # each song is saved as markdown file. The second line contains "title: <title>"
    # we read the title from the file and return a list of AppSong objects
    songs = []

    import os
    for filename in os.listdir(folder):
        if filename.endswith(".md"):
            with open(os.path.join(folder, filename), "r") as f:
                f.readline()
                title = f.readline().strip()
                # remove "title: " from the beginning
                if not title.startswith("title: "):
                    raise Exception("title not found in file " + filename)
                title = title[7:].strip()
                songs.append(AppSong(title, filename))

    return songs

In [70]:
appsongs = read_songs("input/_antwerpse")

In [71]:
appsongs[0].title

'Aan de schachten'

## PlaylistSongs

In [72]:
import pandas
playlistsongs = pandas.read_csv("input/playlist antwerpse codex - overzicht.csv")
# only keep columns "p.", "Titel", "YT url" and "YT titel"
playlistsongs = playlistsongs[["p.", "Titel", "YT url", "YT titel"]]
# drop all rows that have NaN in the "YT url" column
playlistsongs = playlistsongs.dropna(subset=["YT url"])

In [73]:
playlistsongs

,p.,Titel,YT url,YT titel
0,22.0,Salamander,https://youtu.be/5ufTD37fCVc,Salamander
1,37.0,Io vivat,https://youtu.be/aWuI393VMl4,Io Vivat
2,38.0,Gaudeamus igitur,https://youtu.be/SwmVwOHGv8s,Gaudeamus Igitur
3,40.0,Oude roldersklacht,https://youtu.be/uLRQM5pL70I,Oude Roldersklacht
4,42.0,Ruiterslied,https://youtu.be/CGz70cgwnDw,Ruiterslied
...,...,...,...,...
337,469.0,Kermislied,https://youtu.be/NVRZoucx7PI,Micha Marah - Kermis in 't land van Rupelmonde
338,470.0,De duinen,https://youtu.be/77DfJ6Y9rI8,De duinen
339,471.0,Aan het Noordzeestrand,https://youtu.be/arlnBmxt6l0,Will Tura Aan het Noordzeestrand
340,472.0,Tineke van Heule,https://youtu.be/gKIOmry7T78,Tinneke Van Heule (Met Edelweiss-Kapelle o.l.v...


## SongMatcher

In [74]:
# for each song in the playlist, find the corresponding appsong (most similar title)
# and add the youtube link to the appsong
import difflib
for i, row in playlistsongs.iterrows():
    title = row["Titel"]
    yt_link = row["YT url"]
    yt_title = row["YT titel"]
    page = row["p."]

    # find the most similar song
    best_ratio = 0
    best_song = None
    for song in appsongs:
        ratio = difflib.SequenceMatcher(None, title, song.title).ratio()
        if ratio > best_ratio:
            best_ratio = ratio
            best_song = song
    
    if best_ratio > 0.8:
        if best_song.yt_link is not None:
            print("Warning: multiple matches found for", best_song.title)
            print("  ", best_song.yt_link, best_song.yt_title)
            print("  ", yt_link, yt_title)

        best_song.yt_link = yt_link
        best_song.playlist_title = title
        best_song.page = page
    else:
        print("No match found for", title)

No match found for Salamander
No match found for Stad van Diamant
No match found for Ik wil deze nacht in de straten verdwalen


## Check results

In [76]:
for song in appsongs:
    print(f"{song.title} ({song.page}): {song.playlist_title} {song.yt_link}")

Aan de schachten (120.0): Aan de schachten https://youtu.be/NMhVVI2Z270
Aan het Noordzeestrand (471.0): Aan het Noordzeestrand https://youtu.be/arlnBmxt6l0
Abschied (298.0): Abschied https://youtu.be/g5wNWdbgvz8
Abundantia (None): None None
Ach lieflijke meisjes (119.0): Ach lieflijke meisjes https://youtu.be/TtcItcJfWnI
Ade zur guten Nacht (348.0): Ade zur guten Nacht https://youtu.be/EpctqNYm6Uc
Ad fundum per jaar (None): None None
Ad fundum per jaar lidmaatschap (163.0): Ad fundum per jaar lidmaatschap https://youtu.be/n-bP7iqpPZw
Aesculapia (None): None None
Ah la salope (425.0): Ah la salope https://youtu.be/CX1WDgOTtOA
Ai ai ai (448.0): Ai ai ai https://youtu.be/JG2EWB2lBqo
AIESEC (None): None None
Ain boer wol noar zien noaber toe (435.0): Ain boer wol noar zin noaber toe https://youtu.be/h2azGx40524
Ã€ la claire fontaine (414.0): À la claire fontaine https://youtu.be/tJkJe2zMQng
Al boven door het vensterken (134.0): Al boven door het vensterken https://youtu.be/dnkkUYK3FhU
Al d

## Rewriter
Replace for each appsong markdown file "page: (.*)" with the new page and add a youtube link line below it

In [81]:
for song in appsongs:
    if song.yt_link is None:
        print(f"No match found for {song.title} ({song.page})")
        continue

    lines = []
    with open(f"input/_antwerpse/{song.filename}", "r") as f:
        lines = f.readlines()

   # find the index of the "page: (.*)" line
    page_line_index = None
    for i, line in enumerate(lines):
        if line.startswith("page: "):
            page_line_index = i
            break

    if page_line_index is None:
        print(f"Warning: no page line found for {song.title} ({song.page})")
        continue

    # replace the page line with the new page
    lines[page_line_index] = f"page: {int(song.page)}\n"
    # add the youtube link
    lines.insert(page_line_index + 1, f"recording: {song.yt_link}\n")

    with open(f"output/_antwerpse/{song.filename}", "w+") as f:
        f.writelines(lines)

No match found for Abundantia (None)
No match found for Ad fundum per jaar (None)
No match found for Aesculapia (None)
No match found for AIESEC (None)
No match found for ASK-Stuwer (None)
No match found for Biomedica (None)
No match found for Campinaria (None)
No match found for De Chips (None)
No match found for Demetris (None)
No match found for DIEFKA (None)
No match found for EKA (None)
No match found for ESN (None)
No match found for Fabiant (None)
No match found for Heer Halewijn (None)
No match found for Hera (None)
No match found for Het cantusalfabet (None)
No match found for Ik wil deze nacht ... (None)
No match found for IMBIT (None)
No match found for Ingenium (None)
No match found for Jan Rolder (None)
No match found for K.D.A. (None)
No match found for Kinesia (None)
No match found for Klio (None)
No match found for La mÃ¨re Gaspar (None)
No match found for Lingua (None)
No match found for Nordkempus (None)
No match found for Odinn (None)
No match found for Prisma (None)